# ETL Transform: Stocks

This notebook runs the **stocks ETL pipeline**: ingest from Postgres (with warmup window) → transform (returns, volatility, technical indicators) → save to `historical_processed` → publish to S3.

**S3 upload modes (set in config cell below):**
- **Per day**: one CSV per book per day at `stocks/transformed/crypto/book={book}/year=.../month=.../day=.../format=csv/YYYYMMDD-{book}.csv`
- **Batch (run/week/month/year)**: one CSV per run (all books) or per (book, partition) at `stocks/transformed/crypto/book={book}/year=.../week=...` or `month=...` or `year=.../format=csv/...`

Set `AWS_DEFAULT_STOCKS_BUCKET` in `.env` for S3 uploads.

In [1]:
import sys
from pathlib import Path

# Resolve project root: run from repo root or notebooks/etl/
_cwd = Path(".").resolve()
project_root = _cwd if (_cwd / "src").is_dir() else (_cwd.parent.parent if _cwd.name == "etl" else _cwd)
src_path = project_root / "src"
if src_path.is_dir():
    sys.path.insert(0, str(project_root))
    sys.path.insert(0, str(src_path))
else:
    raise FileNotFoundError(f"Expected src at {src_path}. Run from repo root or notebooks/etl/.")

import pandas as pd

In [2]:
# Config: date range, books, and S3 upload options
SINCE = "2026-08-30"
UNTIL = "2026-09-06"
BOOKS = ["btc-usd"]  # None = all books; or e.g. ["btc-usd", "eth-usd"]
WARMUP_DAYS = 252
# S3: per-day (one file per book per day) and/or batch (one file per book per week/month/year)
UPLOAD_S3 = True
UPLOAD_S3_BATCH = ["week"]  # e.g. ["run", "week", "month", "year"] or None

In [3]:
# Run stocks ETL: transform (with warmup for indicators), save to Postgres, publish to S3.

import logging
# Show pipeline progress in the notebook (ingest, transform, save, S3 upload)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S", force=True)
for _name in ("pipelines.etl_transform", "pipelines.etl_cli", "transform.stocks.stock_transformers"):
    logging.getLogger(_name).setLevel(logging.INFO)

from config.settings import get_settings
from pipelines.etl_transform import run_stocks_etl

_settings = get_settings()
_stocks_bucket = _settings.aws.stocks_bucket
if (UPLOAD_S3 or UPLOAD_S3_BATCH) and not _stocks_bucket:
    print(
        "S3 upload skipped: set AWS_DEFAULT_STOCKS_BUCKET in .env, "
        "then restart Jupyter."
    )

transformed_df = run_stocks_etl(
    since=SINCE,
    until=UNTIL,
    books=BOOKS,
    warmup_days=WARMUP_DAYS,
    stocks_bucket=None,  # uses AWS_DEFAULT_STOCKS_BUCKET from .env
    save_to_postgres=False,
    upload_s3=UPLOAD_S3,
    upload_s3_batch=UPLOAD_S3_BATCH,
)

print(f"Transformed {len(transformed_df)} stock records")

14:59:20 - INFO - Ingesting stocks (warmup 2025-12-21 to 2026-09-06)...
14:59:20 - INFO - ============================================================
14:59:20 - INFO - INGEST STOCKS
14:59:20 - INFO - ============================================================
14:59:20 - INFO - Fetching stocks from database...
14:59:20 - INFO -   Books: ['btc-usd']
14:59:20 - INFO - Retrieved 8 records
14:59:20 - INFO - Filtered since 2025-12-21: 8 records
14:59:20 - INFO - Filtered until 2026-09-06: 8 records
14:59:20 - INFO - Ingestion complete: 8 records
14:59:20 - INFO - Transforming (returns, volatility, technical indicators)...
14:59:20 - INFO - Stock transformation pipeline initialized
14:59:20 - INFO - Transforming 8 stock records...
14:59:20 - INFO - Stock transformation complete: 8 records
14:59:20 - INFO - Transformed 8 stock records (2026-08-30 to 2026-09-06)
14:59:20 - INFO - Uploading 8 book/day files to s3://test-financial-stocks-bucket/...


Connection to the database successful!
Table name set to: historical
Connection closed.


14:59:21 - INFO - Uploaded 8 group files to s3://test-financial-stocks-bucket/
14:59:21 - INFO - Uploaded stocks batch week (book=btc-usd, partition 2026-08-30) to s3://test-financial-stocks-bucket/stocks/transformed/crypto/book=btc-usd/year=2026/week=34/format=csv/y2026_w34-btc-usd.csv
14:59:21 - INFO - Uploaded stocks batch week (book=btc-usd, partition 2026-08-31) to s3://test-financial-stocks-bucket/stocks/transformed/crypto/book=btc-usd/year=2026/week=35/format=csv/y2026_w35-btc-usd.csv


Transformed 8 stock records


In [4]:
# Inspect transformed output
if not transformed_df.empty:
    display(transformed_df.head())
    print(transformed_df.columns.tolist())

,ref,book,date,open,high,low,close,adj_close,volume,created_at,...,sma_200,ema_12,ema_26,rsi_14,macd,macd_signal,macd_histogram,bb_upper,bb_middle,bb_lower
0,https://finance.yahoo.com,btc-usd,2026-08-30,78246.17,79373.18,77056.15,77667.57,77667.57,19314839672,2026-09-06 14:26:11.877067,...,NaN,77667.570000,77667.570000,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN
1,https://finance.yahoo.com,btc-usd,2026-08-31,77673.70,79247.34,77378.44,78548.63,78548.63,30608386987,2026-09-06 14:26:11.747547,...,NaN,77803.117692,77732.833704,NaN,70.283989,14.056798,56.227191,NaN,NaN,NaN
2,https://finance.yahoo.com,btc-usd,2026-09-01,78539.86,79196.52,76399.06,77403.63,77403.63,30608796575,2026-09-06 14:26:11.611771,...,NaN,77741.658047,77708.448244,NaN,33.209803,17.887399,15.322404,NaN,NaN,NaN
3,https://finance.yahoo.com,btc-usd,2026-09-02,77402.14,77737.55,76248.30,77300.48,77300.48,26525296646,2026-09-06 14:26:11.501003,...,NaN,77673.784502,77678.228374,NaN,-4.443873,13.421145,-17.865017,NaN,NaN,NaN
4,https://finance.yahoo.com,btc-usd,2026-09-03,77300.16,82262.21,76940.88,81271.74,81271.74,40504199354,2026-09-06 14:26:11.393131,...,NaN,78227.316117,77944.414421,NaN,282.901696,67.317255,215.584441,NaN,NaN,NaN


['ref', 'book', 'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'created_at', 'simple_return', 'log_return', 'volatility_20d', 'volatility_60d', 'volatility_parkinson', 'volatility_gk', 'sma_20', 'sma_50', 'sma_200', 'ema_12', 'ema_26', 'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'bb_upper', 'bb_middle', 'bb_lower']


## Optional: Specific books and date range

In [5]:
# transformed_df = run_stocks_etl(
#     since="2026-01-01",
#     until="2026-01-28",
#     books=["btc-usd", "eth-usd"],
#     warmup_days=252,
#     save_to_postgres=True,
#     upload_s3=True,
# )